# ML Zoomcamp 2026 — Homework 2

Machine Learning for Regression. Dataset dan aturan mengikuti homework resmi 2026. Target: `fuel_efficiency_mpg`.

In [ ]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
FEATURES = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
TARGET = 'fuel_efficiency_mpg'

df = pd.read_csv(DATA_URL)[FEATURES + [TARGET]]
df.head()

## Q1–Q2 — Missing values and median horsepower

In [ ]:
missing = df.isna().sum()
missing[missing > 0], df['horsepower'].median()

Expected: only `horsepower` is missing, and its median is `254.0`.

## Helpers — lecture-style split, linear regression, RMSE

In [ ]:
def split_data(data, seed):
    n = len(data)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    train = data.iloc[idx[:n_train]].reset_index(drop=True)
    val = data.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
    test = data.iloc[idx[n_train + n_val:]].reset_index(drop=True)
    return train, val, test

def prepare_X(frame, fill_value):
    return frame[FEATURES].fillna(fill_value).to_numpy(dtype=float)

def train_linear_regression(X, y, r=0.0):
    X = np.column_stack([np.ones(len(X)), X])
    XTX = X.T @ X
    reg = r * np.eye(XTX.shape[0])
    reg[0, 0] = 0.0
    w = np.linalg.solve(XTX + reg, X.T @ y)
    return w[0], w[1:]

def predict(X, w0, w):
    return w0 + X @ w

def rmse(y, y_pred):
    return np.sqrt(np.mean((y - y_pred) ** 2))

## Q3 — Zero vs training-mean imputation

In [ ]:
df_train, df_val, df_test = split_data(df, seed=42)
y_train = df_train[TARGET].to_numpy()
y_val = df_val[TARGET].to_numpy()

mean_hp = df_train['horsepower'].mean()
scores_q3 = {}
for label, fill_value in [('With 0', 0.0), ('With mean', mean_hp)]:
    X_train = prepare_X(df_train, fill_value)
    X_val = prepare_X(df_val, fill_value)
    w0, w = train_linear_regression(X_train, y_train)
    scores_q3[label] = round(rmse(y_val, predict(X_val, w0, w)), 3)
scores_q3

Expected: `{'With 0': 2.205, 'With mean': 2.202}` → **With mean**.

## Q4 — Regularization

In [ ]:
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]
X_train = prepare_X(df_train, 0.0)
X_val = prepare_X(df_val, 0.0)
scores_q4 = {}
for r in r_values:
    w0, w = train_linear_regression(X_train, y_train, r=r)
    scores_q4[r] = round(rmse(y_val, predict(X_val, w0, w)), 4)
scores_q4

All options round to `2.2053`; per homework tie rule choose the smallest `r`: **0**.

## Q5 — Stability across seeds

In [ ]:
seed_scores = []
for seed in range(10):
    train, val, _ = split_data(df, seed=seed)
    X_train = prepare_X(train, 0.0)
    X_val = prepare_X(val, 0.0)
    y_train = train[TARGET].to_numpy()
    y_val = val[TARGET].to_numpy()
    w0, w = train_linear_regression(X_train, y_train)
    seed_scores.append(rmse(y_val, predict(X_val, w0, w)))

seed_scores, round(np.std(seed_scores), 3)

Expected standard deviation: **0.029**.

## Q6 — Train+validation, seed 9, r=0.001

In [ ]:
train, val, test = split_data(df, seed=9)
train_full = pd.concat([train, val], ignore_index=True)

X_train_full = prepare_X(train_full, 0.0)
X_test = prepare_X(test, 0.0)
y_train_full = train_full[TARGET].to_numpy()
y_test = test[TARGET].to_numpy()

w0, w = train_linear_regression(X_train_full, y_train_full, r=0.001)
score_q6 = rmse(y_test, predict(X_test, w0, w))
round(score_q6, 3)

Expected test RMSE: **2.236**.

### Final submission answers

| Question | Answer |
|---|---|
| Q1 | `horsepower` |
| Q2 | `254` |
| Q3 | `With mean` |
| Q4 | `0` |
| Q5 | `0.029` |
| Q6 | `2.236` |